# WEEK-07: NAÏVE BAYES CLASSIFIER

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import csv
import math
from collections import defaultdict, Counter

1.<br> Implement in python program of the following problems using Bayes Theorem.
<br>a) Of the students in the college, 60% of the students reside in the hostel and 40% of the students are day
scholars. Previous year results report that 30% of all students who stay in the hostel scored A Grade and 20%
of day scholars scored A grade. At the end of the year, one student is chosen at random and found that he/she
has an A grade. What is the probability that the student is a hosteler?
<br>b) Suppose you're testing for a rare disease, and you have the following information:
<br> The disease has a prevalence of 0.01 (1% of the population has the disease).
<br> The test is not perfect:
<br> The test correctly identifies the disease (true positive) 99% of the time (sensitivity).
<br> The test incorrectly indicates the disease (false positive) 2% of the time (1 - specificity).
<br>Calculate the probability of having the disease given a positive test result using Bayes' theorem.

In [3]:
#A

#given
P_hosteler = 0.60
P_day_scholar = 0.40

P_A_given_hosteler = 0.30
P_A_given_day_scholar = 0.20

P_A = (P_A_given_hosteler * P_hosteler) + \
      (P_A_given_day_scholar * P_day_scholar)

#Bayes theorem
P_hosteler_given_A = (P_A_given_hosteler * P_hosteler) / P_A

print("Probability that the A-grade student is a hosteler:", P_hosteler_given_A)
print("Percentage:",P_hosteler_given_A * 100, "%")


Probability that the A-grade student is a hosteler: 0.6923076923076923
Percentage: 69.23076923076923 %


In [4]:
#B
#given
P_disease = 0.01
P_no_disease = 1 - P_disease

P_positive_given_disease = 0.99
P_positive_given_no_disease = 0.02

P_positive = (P_positive_given_disease * P_disease + P_positive_given_no_disease * P_no_disease)

# Bayes theorem
P_disease_given_positive = (P_positive_given_disease * P_disease) / P_positive

print("Probability of having the disease given a positive test:",P_disease_given_positive)
print("Percentage:",P_disease_given_positive * 100, "%")


Probability of having the disease given a positive test: 0.3333333333333333
Percentage: 33.33333333333333 %


2.<br>Develop a function python code for Naïve Bayes classifier from scratch without using scikit-learn library,
to predict whether the buyer should buy computer or not. Consider a following sample training dataset stored
in a CSV file containing information about following buyer conditions (such as “<=30,” “medium,” “Yes,”
and “fair”) and whether the player played golf (“Yes” or “No”).

In [11]:
class NaiveBayes:

    def __init__(self):
        self.classes = []
        self.prior = {}
        self.conditional_prob = {}
        self.features = []
        self.feature_values = {}
        self.total_rows = 0

    def fit(self, data, target):
        self.total_rows = len(data)
        self.features = [
            column for column in data[0]
            if column != target
        ]
        self.classes = list(
            set(row[target] for row in data)
        )
        for feature in self.features:
            self.feature_values[feature] = set(
                row[feature] for row in data
            )
        for cls in self.classes:

            count = sum(
                1 for row in data
                if row[target] == cls
            )
            self.prior[cls] = count / self.total_rows
        for cls in self.classes:

            self.conditional_prob[cls] = {}

            class_rows = [
                row for row in data
                if row[target] == cls
            ]

            class_count = len(class_rows)

            for feature in self.features:

                self.conditional_prob[cls][feature] = {}

                total_values = len(
                    self.feature_values[feature]
                )

                for value in self.feature_values[feature]:

                    value_count = sum(
                        1 for row in class_rows
                        if row[feature] == value
                    )
                    probability = (
                        value_count + 1
                    ) / (
                        class_count + total_values
                    )

                    self.conditional_prob[cls][feature][value] = probability

    # Predict for one sample
    def predict(self, sample):
        probabilities = {}
        for cls in self.classes:
            probability = self.prior[cls]
            for feature in self.features:
                value = sample[feature]
                if value in self.conditional_prob[cls][feature]:
                    probability *= (
                        self.conditional_prob
                        [cls][feature][value]
                    )
                else:
                    class_count = round(
                        self.prior[cls] * self.total_rows)

                    total_values = len(
                        self.feature_values[feature])

                    probability *= 1 / (
                        class_count + total_values)
            probabilities[cls] = probability

        prediction = max(probabilities,key=probabilities.get)
        return prediction, probabilities


#main
def read_csv(filename):
    with open(filename, "r") as file:
        reader = csv.DictReader(file)
        return list(reader)


data = read_csv("computer.csv")
model = NaiveBayes()
model.fit(data, "computer")

sample = {
    "age": "<=30",
    "income": "medium",
    "student": "yes",
    "credit_rating": "fair"
}

prediction, probabilities = model.predict(sample)
print("Buyer details:")
print(sample)
print("\nProbability scores:")
for cls, probability in probabilities.items():
    print("P(computer =", cls, ") =", probability)
    
print("\nFinal Prediction:", prediction)

if prediction == "yes":
    print("The buyer should buy the computer.")
else:
    print("The buyer should NOT buy the computer.")


Buyer details:
{'age': '<=30', 'income': 'medium', 'student': 'yes', 'credit_rating': 'fair'}

Probability scores:
P(computer = no ) = 0.008199708454810493
P(computer = yes ) = 0.027117768595041326

Final Prediction: yes
The buyer should buy the computer.


3.<br>Write a Python function to implement the Naive Bayes classifier without using the scikit-learn library for the
following sample training dataset stored as a .CSV file.
<br>a. Build a classifier that determines whether a text is about sports or not.
<br>b. Determine which tag the sentence "A very close game" belongs to

In [16]:
class NaiveBayesTextClassifier:
    def __init__(self, smoothing=1.0):
        self.smoothing = smoothing
        self.class_word_counts = defaultdict(Counter)
        self.class_doc_counts = Counter()
        self.vocab = set()
        self.priors = {}
        self.total_docs = 0

    def preprocess(self, text):
        return text.lower().split()

    def train(self, dataset):
        self.total_docs = len(dataset)
        
        for text, label in dataset:
            words = self.preprocess(text)
            self.class_doc_counts[label] += 1
            for word in words:
                self.class_word_counts[label][word] += 1
                self.vocab.add(word)
                
        for label, count in self.class_doc_counts.items():
            self.priors[label] = count / self.total_docs

    def predict(self, text):
        words = self.preprocess(text)
        vocab_size = len(self.vocab)
        log_probabilities = {}
        
        for label in self.priors:
            score = math.log(self.priors[label])
            total_words_in_class = sum(self.class_word_counts[label].values())
            
            for word in words:
                word_count = self.class_word_counts[label][word]
                word_prob = (word_count + self.smoothing) / (total_words_in_class + (self.smoothing * vocab_size))
                score += math.log(word_prob)
                
            log_probabilities[label] = score
        best_label = max(log_probabilities, key=log_probabilities.get)
        return best_label, log_probabilities

training_data = [
    ("A great game", "Sports"),
    ("The election was over", "Not sports"),
    ("Very clean match", "Sports"),
    ("A clean but forgettable game", "Sports"),
    ("It was a close election", "Not sports")
]

classifier = NaiveBayesTextClassifier(smoothing=1.0)
classifier.train(training_data)

test_sentence = "A very close game"
predicted_tag, scores = classifier.predict(test_sentence)

print(f"Test Sentence: '{test_sentence}'")
print(f"Predicted Tag: *{predicted_tag}*")
print("\nLog-likelihood scores:")
for label, score in scores.items():
    print(f" - {label}: {score:.4f}")

Test Sentence: 'A very close game'
Predicted Tag: *Sports*

Log-likelihood scores:
 - Sports: -10.4960
 - Not sports: -12.0720
